In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
import re


# ============================================================
# TEAM 3 - AIRFLOW-CONTROLLED INCREMENTAL BRONZE INGESTION
#
# Airflow passes the exact batch to process:
#
#     input_batch = batch_NNN
#
# Bronze:
#
# 1. Reads input_batch from Databricks job parameters
# 2. Validates batch_NNN format
# 3. Validates the batch directory exists
# 4. Validates ALL 4 required CSV files exist
# 5. Processes ONLY the Airflow-selected batch
# 6. Adds Bronze metadata
# 7. Performs insert-only Delta MERGEs
#
# Bronze DOES NOT independently discover the latest batch.
#
# Example:
#
# Airflow detects:
#     batch_009
#
# Airflow passes:
#     input_batch = batch_009
#
# Bronze processes:
#     landing/batch_009/
#
# ============================================================


# ------------------------------------------------------------
# 1. JOB PARAMETERS
# ------------------------------------------------------------

dbutils.widgets.text(
    "input_batch",
    "",
    "Input Batch",
)

INPUT_BATCH = (
    dbutils.widgets
    .get("input_batch")
    .strip()
)


# ------------------------------------------------------------
# 2. CONFIGURATION
# ------------------------------------------------------------

CATALOG = "dbwteam3"
SCHEMA = "order_fulfillment"

TARGET = f"{CATALOG}.{SCHEMA}"

BASE_PATH = (
    "abfss://team3@adlsteam3.dfs.core.windows.net/landing"
)


# ------------------------------------------------------------
# 3. DATASET CONFIGURATION
# ------------------------------------------------------------

DATASETS = {

    "orders": {
        "file": "orders.csv",
        "table": f"{TARGET}.bronze_orders",
        "keys": ["order_id"],
    },

    "order_lines": {
        "file": "order_lines.csv",
        "table": f"{TARGET}.bronze_order_lines",
        "keys": [
            "order_id",
            "line_id",
        ],
    },

    "shipment_lines": {
        "file": "shipment_lines.csv",
        "table": f"{TARGET}.bronze_shipment_lines",
        "keys": [
            "shipment_id",
            "order_id",
            "line_id",
        ],
    },

    "delivery_events": {
        "file": "delivery_events.csv",
        "table": f"{TARGET}.bronze_delivery_events",
        "keys": ["event_id"],
    },
}


REQUIRED_FILES = {
    "orders.csv",
    "order_lines.csv",
    "shipment_lines.csv",
    "delivery_events.csv",
}


METADATA_COLUMNS = {
    "_source_file",
    "_ingestion_timestamp",
    "_batch_id",
}


# ------------------------------------------------------------
# 4. VALIDATE AIRFLOW INPUT BATCH
# ------------------------------------------------------------

print("=" * 70)
print("PHASE 3 - AIRFLOW-CONTROLLED BRONZE INGESTION")
print("=" * 70)

print("Catalog     :", CATALOG)
print("Schema      :", SCHEMA)
print("Landing     :", BASE_PATH)
print("Input batch :", INPUT_BATCH)

print("=" * 70)


# input_batch MUST be supplied by Airflow
if not INPUT_BATCH:

    raise ValueError(
        "Required Databricks job parameter "
        "'input_batch' was not provided."
    )


# Strict batch_NNN format
batch_pattern = re.compile(
    r"^batch_(\d{3})$"
)


match = batch_pattern.fullmatch(
    INPUT_BATCH
)


if not match:

    raise ValueError(
        "Invalid input_batch. "
        f"Received: {INPUT_BATCH!r}. "
        "Expected format: batch_NNN "
        "(example: batch_009)."
    )


SELECTED_BATCH_NUMBER = int(
    match.group(1)
)

SELECTED_BATCH_ID = INPUT_BATCH


SELECTED_BATCH_PATH = (
    f"{BASE_PATH}/{SELECTED_BATCH_ID}"
)


print()
print(
    "Airflow selected batch :",
    SELECTED_BATCH_ID,
)

print(
    "Batch number           :",
    SELECTED_BATCH_NUMBER,
)

print(
    "Selected batch path    :",
    SELECTED_BATCH_PATH,
)


# ------------------------------------------------------------
# 5. VERIFY SELECTED BATCH DIRECTORY EXISTS
# ------------------------------------------------------------

try:

    batch_entries = (
        dbutils.fs.ls(
            SELECTED_BATCH_PATH
        )
    )

except Exception as exc:

    raise ValueError(
        "The Airflow-selected batch directory "
        "does not exist or cannot be accessed: "
        f"{SELECTED_BATCH_PATH}"
    ) from exc


# ------------------------------------------------------------
# 6. VERIFY ALL REQUIRED FILES EXIST
#
# batch_009 onward contract:
#
#     orders.csv
#     order_lines.csv
#     shipment_lines.csv
#     delivery_events.csv
#
# landing/reference/pharmacies.csv is NOT part
# of the incremental batch contract.
# ------------------------------------------------------------

actual_files = {
    entry.name.rstrip("/")
    for entry in batch_entries
    if not entry.name.endswith("/")
}


print()
print(
    "Files found in selected batch:"
)

for file_name in sorted(actual_files):
    print(
        " -",
        file_name,
    )


missing_files = (
    REQUIRED_FILES
    - actual_files
)


if missing_files:

    raise ValueError(
        f"{SELECTED_BATCH_ID} is incomplete. "
        "Missing required files: "
        f"{sorted(missing_files)}"
    )


print()
print(
    "Required-file validation passed."
)

print(
    "All 4 required input files are present."
)

print("=" * 70)


# ------------------------------------------------------------
# 7. INGEST ONE DATASET
# ------------------------------------------------------------

def ingest_dataset(
    dataset_name,
    config,
):

    print()
    print("=" * 70)

    print(
        f"PROCESSING: {dataset_name}"
    )

    print("=" * 70)


    file_name = config["file"]

    target_table = config["table"]

    merge_keys = config["keys"]


    source_path = (
        f"{SELECTED_BATCH_PATH}/{file_name}"
    )


    print(
        "Batch :",
        SELECTED_BATCH_ID,
    )

    print(
        "Source:",
        source_path,
    )

    print(
        "Target:",
        target_table,
    )


    # --------------------------------------------------------
    # A. CHECK TARGET BRONZE TABLE
    # --------------------------------------------------------

    if not spark.catalog.tableExists(
        target_table
    ):

        raise ValueError(
            "Target Bronze table does not exist: "
            f"{target_table}"
        )


    # --------------------------------------------------------
    # B. LOAD TARGET TABLE
    # --------------------------------------------------------

    target_df = spark.table(
        target_table
    )


    target_columns = (
        target_df.columns
    )


    # --------------------------------------------------------
    # C. CHECK REQUIRED METADATA COLUMNS
    # --------------------------------------------------------

    missing_metadata = (
        METADATA_COLUMNS
        - set(target_columns)
    )


    if missing_metadata:

        raise ValueError(
            f"{target_table} is missing metadata "
            f"columns: {sorted(missing_metadata)}"
        )


    # --------------------------------------------------------
    # D. CHECK MERGE KEYS
    # --------------------------------------------------------

    missing_keys = (
        set(merge_keys)
        - set(target_columns)
    )


    if missing_keys:

        raise ValueError(
            f"{target_table} is missing merge keys: "
            f"{sorted(missing_keys)}"
        )


    # --------------------------------------------------------
    # E. GET BUSINESS SCHEMA FROM TARGET
    # --------------------------------------------------------

    business_schema = (
        target_df
        .drop(
            *METADATA_COLUMNS
        )
        .schema
    )


    # --------------------------------------------------------
    # F. READ ONLY THE AIRFLOW-SELECTED BATCH
    #
    # Example:
    #
    # input_batch=batch_009
    #
    # Reads:
    #
    # landing/batch_009/orders.csv
    #
    # It does NOT discover another batch.
    # --------------------------------------------------------

    raw_source_df = (

        spark.read

        .format("csv")

        .option(
            "header",
            "true"
        )

        .schema(
            business_schema
        )

        .load(
            source_path
        )
    )


    # --------------------------------------------------------
    # G. ADD BRONZE METADATA
    # --------------------------------------------------------

    source_df = (

        raw_source_df

        .select(

            "*",

            F.col(
                "_metadata.file_path"
            ).alias(
                "_source_file"
            ),

        )

        .withColumn(
            "_ingestion_timestamp",
            F.current_timestamp(),
        )

        .withColumn(
            "_batch_id",
            F.regexp_extract(
                F.col("_source_file"),
                r"/(batch_\d{3})/",
                1,
            )
        )

    )


    # --------------------------------------------------------
    # H. VERIFY EXTRACTED BATCH
    #
    # Airflow selected batch_NNN.
    # Every row MUST originate from that same batch.
    # --------------------------------------------------------

    wrong_batch_count = (

        source_df

        .filter(
            F.col("_batch_id")
            != SELECTED_BATCH_ID
        )

        .count()
    )


    if wrong_batch_count:

        raise ValueError(
            f"{dataset_name}: "
            f"{wrong_batch_count} rows did not "
            f"originate from the Airflow-selected "
            f"batch {SELECTED_BATCH_ID}."
        )


    # --------------------------------------------------------
    # I. COUNT SOURCE ROWS
    # --------------------------------------------------------

    source_count = (
        source_df.count()
    )


    print(
        "Source rows:",
        source_count,
    )


    if source_count == 0:

        raise ValueError(
            f"{dataset_name}: "
            f"{source_path} contains no records."
        )


    # --------------------------------------------------------
    # J. VALIDATE MERGE KEYS
    # --------------------------------------------------------

    invalid_condition = None


    for key in merge_keys:

        condition = (

            F.col(key).isNull()

            |

            (
                F.trim(
                    F.col(key)
                    .cast("string")
                )
                == ""
            )

        )


        if invalid_condition is None:

            invalid_condition = (
                condition
            )

        else:

            invalid_condition = (
                invalid_condition
                | condition
            )


    invalid_count = (

        source_df

        .filter(
            invalid_condition
        )

        .count()

    )


    if invalid_count:

        print(
            f"Invalid rows found in "
            f"{SELECTED_BATCH_ID}: "
            f"{invalid_count}"
        )


        source_df.filter(
            invalid_condition
        ).show(
            truncate=False
        )


        raise ValueError(

            f"{dataset_name} in "
            f"{SELECTED_BATCH_ID} contains "
            f"{invalid_count} rows with "
            f"missing merge keys."

        )


    print(
        "Merge-key validation passed."
    )


    # --------------------------------------------------------
    # K. DEDUPLICATE SELECTED BATCH
    # --------------------------------------------------------

    source_df = (

        source_df

        .dropDuplicates(
            merge_keys
        )

    )


    distinct_source_count = (
        source_df.count()
    )


    print(
        "Distinct source rows:",
        distinct_source_count,
    )


    # --------------------------------------------------------
    # L. BUILD MERGE CONDITION
    # --------------------------------------------------------

    merge_condition = " AND ".join(

        [

            f"target.{key} = source.{key}"

            for key
            in merge_keys

        ]

    )


    print(
        "Merge condition:",
        merge_condition,
    )


    # --------------------------------------------------------
    # M. COUNT TARGET BEFORE MERGE
    # --------------------------------------------------------

    before_count = (
        target_df.count()
    )


    # --------------------------------------------------------
    # N. INSERT-ONLY DELTA MERGE
    #
    # Existing key:
    #       no change
    #
    # New key:
    #       insert
    #
    # This makes replay of the same batch safer.
    # --------------------------------------------------------

    delta_target = (

        DeltaTable.forName(
            spark,
            target_table,
        )

    )


    (

        delta_target

        .alias(
            "target"
        )

        .merge(

            source_df.alias(
                "source"
            ),

            merge_condition,

        )

        .whenNotMatchedInsertAll()

        .execute()

    )


    # --------------------------------------------------------
    # O. POST-MERGE COUNTS
    # --------------------------------------------------------

    after_count = (

        spark

        .table(
            target_table
        )

        .count()

    )


    inserted_count = (
        after_count
        - before_count
    )


    print()

    print(
        f"{dataset_name} ingestion completed."
    )

    print(
        f"Batch    : {SELECTED_BATCH_ID}"
    )

    print(
        f"Before   : {before_count}"
    )

    print(
        f"Source   : {distinct_source_count}"
    )

    print(
        f"Inserted : {inserted_count}"
    )

    print(
        f"After    : {after_count}"
    )


    # --------------------------------------------------------
    # P. RETURN RESULT
    # --------------------------------------------------------

    return (

        dataset_name,

        SELECTED_BATCH_ID,

        distinct_source_count,

        inserted_count,

        after_count,

    )


# ------------------------------------------------------------
# 8. PROCESS SELECTED BATCH IN REFERENTIAL ORDER
#
# Order
#    ↓
# Order Line
#    ↓
# Shipment Line
#    ↓
# Delivery Event
# ------------------------------------------------------------

PROCESSING_ORDER = [

    "orders",

    "order_lines",

    "shipment_lines",

    "delivery_events",

]


results = []


for dataset_name in PROCESSING_ORDER:

    result = ingest_dataset(

        dataset_name,

        DATASETS[
            dataset_name
        ],

    )


    results.append(
        result
    )


# ------------------------------------------------------------
# 9. INGESTION SUMMARY
# ------------------------------------------------------------

verification_df = (

    spark.createDataFrame(

        results,

        [

            "dataset",

            "batch_id",

            "source_rows",

            "inserted_rows",

            "target_rows",

        ],

    )
)


print()
print("=" * 70)

print(
    "AIRFLOW-SELECTED BRONZE INGESTION SUMMARY"
)

print("=" * 70)


display(
    verification_df
)


# ------------------------------------------------------------
# 10. VERIFY ONLY AIRFLOW-SELECTED BATCH
# ------------------------------------------------------------

print()
print("=" * 70)

print(
    f"VERIFYING {SELECTED_BATCH_ID}"
)

print("=" * 70)


for dataset_name, config in DATASETS.items():

    print()

    print(
        f"Dataset: {dataset_name}"
    )


    (

        spark

        .table(
            config["table"]
        )

        .filter(
            F.col("_batch_id")
            == SELECTED_BATCH_ID
        )

        .groupBy(
            "_batch_id"
        )

        .count()

        .show()

    )


# ------------------------------------------------------------
# 11. SHOW SELECTED BATCH ORDERS
# ------------------------------------------------------------

print()
print("=" * 70)

print(
    f"{SELECTED_BATCH_ID} - ORDERS"
)

print("=" * 70)


(

    spark

    .table(
        f"{TARGET}.bronze_orders"
    )

    .filter(
        F.col("_batch_id")
        == SELECTED_BATCH_ID
    )

    .orderBy(
        "order_id"
    )

    .show(
        truncate=False
    )

)


# ------------------------------------------------------------
# 12. SHOW SELECTED BATCH ORDER LINES
# ------------------------------------------------------------

print()
print("=" * 70)

print(
    f"{SELECTED_BATCH_ID} - ORDER LINES"
)

print("=" * 70)


(

    spark

    .table(
        f"{TARGET}.bronze_order_lines"
    )

    .filter(
        F.col("_batch_id")
        == SELECTED_BATCH_ID
    )

    .orderBy(
        "order_id",
        "line_id",
    )

    .show(
        truncate=False
    )

)


# ------------------------------------------------------------
# 13. SHOW SELECTED BATCH SHIPMENT LINES
# ------------------------------------------------------------

print()
print("=" * 70)

print(
    f"{SELECTED_BATCH_ID} - SHIPMENT LINES"
)

print("=" * 70)


(

    spark

    .table(
        f"{TARGET}.bronze_shipment_lines"
    )

    .filter(
        F.col("_batch_id")
        == SELECTED_BATCH_ID
    )

    .orderBy(
        "shipment_id",
        "line_id",
    )

    .show(
        truncate=False
    )

)


# ------------------------------------------------------------
# 14. SHOW SELECTED BATCH DELIVERY EVENTS
# ------------------------------------------------------------

print()
print("=" * 70)

print(
    f"{SELECTED_BATCH_ID} - DELIVERY EVENTS"
)

print("=" * 70)


(

    spark

    .table(
        f"{TARGET}.bronze_delivery_events"
    )

    .filter(
        F.col("_batch_id")
        == SELECTED_BATCH_ID
    )

    .orderBy(
        "event_id"
    )

    .show(
        truncate=False
    )

)


# ------------------------------------------------------------
# 15. FINAL MESSAGE
# ------------------------------------------------------------

print()
print("=" * 70)

print(
    "PHASE 3 AIRFLOW-CONTROLLED BRONZE "
    "INGESTION COMPLETED SUCCESSFULLY."
)

print(
    "Processed batch:",
    SELECTED_BATCH_ID,
)

print("=" * 70)

Bronze incremental delivery-event ingestion completed.
+----------+----------------+
|total_rows|unique_event_ids|
+----------+----------------+
|      6230|            6230|
+----------+----------------+

+---------+-----+
|_batch_id|count|
+---------+-----+
|batch_001| 5280|
|batch_002|  459|
|batch_003|  459|
|batch_004|   30|
|batch_005|    2|
+---------+-----+

